In [15]:
import yfinance as yf
import pandas as pd

In [16]:
#Download 3 years of daily historical data for NVIDIA
df=yf.download("NVDA", start="2023-01-01", end="2026-09-14",interval="1d",multi_level_index=False)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

[*********************100%***********************]  1 of 1 completed


In [17]:
import  talib #Assuming standard wrapper setup

close_prices = df["Close"].values.flatten()  # Flatten the array to 1D if necessary
#calculate a 50-day SMA and a 14-period RSI
df["SMA_50"]=talib.SMA(close_prices, timeperiod=50)
df["RSI_14"]=talib.RSI(close_prices, timeperiod=14)
#Drop missing values caused by indicator lag windows
df.dropna(inplace=True)

In [21]:
import numpy as np #for mathematical signal generation and returns calculations
#re verify clean 1D series
close_series = df["Close"].iloc[:, 0] if len(df["Close"].shape) > 1 else df["Close"]

#Generate continuous binary trading positions(1=Holdlong,0=cash)
df["Signal"]=np.where((close_series > df["SMA_50"]) & (df["RSI_14"] < 65),1,0)
#Calculate daily asset log returns
df["Market_Returns"]=np.log(close_series / close_series.shift(1))
#Calculate strategy returns by multiplying market returns by the previous day's signal
df["Strategy_Returns"]=df["Market_Returns"] * df["Signal"].shift(1) #Shift signal to avoid lookahead bias

In [24]:
import vectorbt as vbt #for backtesting and evaluation
#Isolate the exact 1D series vectors for vectorbt mapping
vbt_close = df["Close"].iloc[:, 0] if len(df["Close"].shape) > 1 else df["Close"]
vbt_signals = df["Signal"].iloc[:, 0] if len(df["Signal"].shape) > 1 else df["Signal"]
#Run the simulation assuming a $10,000 initial capital and no transaction costs
portfolio=vbt.Portfolio.from_signals(vbt_close, entries=(vbt_signals==1), exits=(vbt_signals==0),init_cash=10000,freq="d")
#print metrics like sharpe ratio, max drawdown, and total return
print(portfolio.stats())

Start                               2023-03-15 00:00:00
End                                 2026-09-11 00:00:00
Period                                877 days 00:00:00
Start Value                                     10000.0
End Value                                  16444.990144
Total Return [%]                              64.449901
Benchmark Return [%]                            803.841
Max Gross Exposure [%]                            100.0
Total Fees Paid                                     0.0
Max Drawdown [%]                              31.607021
Max Drawdown Duration                 238 days 00:00:00
Total Trades                                         67
Total Closed Trades                                  66
Total Open Trades                                     1
Open Trade PnL                               895.057616
Win Rate [%]                                  48.484848
Best Trade [%]                                23.761748
Worst Trade [%]                              -15

In [ ]:
print(df[["Close", "SMA_50"]].head(60))  # Look at the first 60 rows
print("\nMissing values count:")
print(df[["Close", "SMA_50"]].isna().sum())
print("\nTotal dataset rows:", len(df))


                Close     SMA_50
Date                            
2023-03-15  24.151371        NaN
2023-03-16  25.460211        NaN
2023-03-17  25.643633        NaN
2023-03-20  25.818075        NaN
2023-03-21  26.116133        NaN
2023-03-22  26.384279        NaN
2023-03-23  27.104998        NaN
2023-03-24  26.694298        NaN
2023-03-27  26.447084        NaN
2023-03-28  26.326462        NaN
2023-03-29  26.898647        NaN
2023-03-30  27.296389        NaN
2023-03-31  27.689140        NaN
2023-04-03  27.876549        NaN
2023-04-04  27.366165        NaN
2023-04-05  26.795979        NaN
2023-04-06  26.951481        NaN
2023-04-10  27.491770        NaN
2023-04-11  27.083065        NaN
2023-04-12  26.411194        NaN
2023-04-13  26.379297        NaN
2023-04-14  26.673359        NaN
2023-04-17  26.916595        NaN
2023-04-18  27.579493        NaN
2023-04-19  27.842655        NaN
2023-04-20  27.018270        NaN
2023-04-21  27.033220        NaN
2023-04-24  26.956467        NaN
2023-04-25

In [25]:
import talib
# 1. Flatten df["Close"] to a 1D series so TA-Lib can read it
close_prices = df["Close"]


# Calculate the 50-day simple moving average
df["SMA_50"] = talib.SMA(close_prices, timeperiod=50)

# Drop rows where SMA hasn't started calculating yet
df_clean = df.dropna(subset=["SMA_50"])

In [26]:
import plotly.graph_objects as go #for interactive plotting
fig=go.Figure()
#Add the close price trace
fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["Close"], mode="lines", name="NVDAClose Price", line=dict(color="#76B900")))
#Add the 50-day SMA trace
fig.add_trace(go.Scatter(x=df_clean.index, y=df_clean["SMA_50"], mode="lines", name="50-Day SMA", line=dict(color="#1f77b4", dash="dash")))
fig.update_layout(
    title="NVIDIA Corporation (NVDA) - Close Price vs 50-Day SMA",
    xaxis_title="Date",
    yaxis_title="Price (USD)",
    template="plotly_white",
    hovermode="x unified"
)
fig.show()

In [28]:
# 1. Cleanly plot the equity curve subplots (handles both value and benchmark automatically)
fig = portfolio.plot(
    subplots=['value'] # Keeps it cleanly on a single graph without broken layout parameters
)

# 2. Update layout titles for clarity
fig.update_layout(
    title='Strategy Equity Curve vs. Benchmark (Buy & Hold)',
    xaxis_title='Date',
    yaxis_title='Portfolio Value ($)',
    template='plotly_white'
)

# 3. Render the interactive chart
fig.show()
